# 18-18 · Живое превью: coords()

Практика к разделу [«Живое превью: coords()»](../../site/chapters/glava-18/18-18-zhivoe-prevyu.html).

## Цель

CREATE ONCE → UPDATE MANY TIMES → COMMIT для прямоугольника-превью.

## Про mainloop() и события мыши в этом ноутбуке

Вместо `root.mainloop()` здесь используется `root.update()` + `root.destroy()`. А вместо настоящих кликов мышью мы вызываем функции-обработчики напрямую с маленьким объектом `FakeEvent(x, y)` вместо настоящего события — Tkinter в реальном приложении передаёт объект с такими же полями `.x`/`.y`, так что код обработчиков проверяется по-настоящему.

In [1]:
class FakeEvent:
    def __init__(self, x, y):
        self.x = x
        self.y = y


## Рабочий пример

In [2]:
import tkinter as tk

root = tk.Tk()
canvas = tk.Canvas(root, width=300, height=200, bg="white")
canvas.pack()

start_x = start_y = None
preview_id = None

def on_press(event):
    global start_x, start_y, preview_id
    start_x, start_y = event.x, event.y
    preview_id = canvas.create_rectangle(event.x, event.y, event.x, event.y, outline="blue", dash=(4, 2))

def on_drag(event):
    canvas.coords(preview_id, start_x, start_y, event.x, event.y)  # ОДИН и тот же элемент

canvas.bind("<Button-1>", on_press)
canvas.bind("<B1-Motion>", on_drag)
root.update()

on_press(FakeEvent(20, 20))
posle_nazhatiya = len(canvas.find_all())
print("Элементов на холсте после нажатия:", posle_nazhatiya)

for x in range(40, 200, 20):          # восемь движений мыши подряд
    on_drag(FakeEvent(x, x))
posle_vosmi_dvizhenij = len(canvas.find_all())
print("Элементов после восьми движений:", posle_vosmi_dvizhenij)
print("Координаты превью:", canvas.coords(preview_id))
root.destroy()


Элементов на холсте после нажатия: 1
Элементов после восьми движений: 1
Координаты превью: [20.0, 20.0, 180.0, 180.0]


## Проверка результата

In [3]:
assert posle_nazhatiya == 1, "нажатие должно создать РОВНО один элемент-превью"
assert posle_vosmi_dvizhenij == 1, (
    "coords() обновляет существующий элемент — после восьми движений "
    "элемент должен остаться ОДИН, а не превратиться в девять"
)
print("CREATE ONCE -> UPDATE MANY TIMES: один элемент пережил восемь движений мыши.")


CREATE ONCE -> UPDATE MANY TIMES: один элемент пережил восемь движений мыши.


## Задание ★★ Самостоятельная задача

Подтвердите на реальном окне, что во время долгого перетаскивания количество элементов на холсте (`len(canvas.find_all())`) не растёт — потому что coords() обновляет существующий элемент, а не создаёт новые.